# PlantDoctor - train the full 10-class tomato model

Run this on **Colab with a GPU runtime** (Runtime > Change runtime type > T4 GPU).
Your laptop is CPU-only, so training happens here instead.

This notebook:
1. Downloads just the tomato classes from the public [PlantVillage dataset](https://github.com/spMohanty/PlantVillage-Dataset) (git sparse-checkout, so we don't pull all 38 crop/disease classes).
2. Fine-tunes a MobileNetV2 classifier (10 classes: healthy + 9 diseases).
3. Converts the trained model to TF.js format, matching what `App.js` expects in `public/model/`.

Run all cells top to bottom, then follow the download step at the end.

## 1. Get the data (tomato classes only)

In [ ]:
!git clone --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git
%cd PlantVillage-Dataset
!git sparse-checkout init --no-cone
!git sparse-checkout set "raw/color/Tomato___*"
%cd ..

In [ ]:
import pathlib

DATA_DIR = pathlib.Path("PlantVillage-Dataset/raw/color")
classes = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
print(len(classes), "classes:")
for c in classes:
    n = len(list((DATA_DIR / c).glob("*.JPG"))) + len(list((DATA_DIR / c).glob("*.jpg")))
    print(f"  {c}: {n} images")

You should see exactly 10 `Tomato___*` classes (healthy + 9 diseases), matching
the classes in `C:\Users\dell\Desktop\testing` used to validate this model.
If the count looks off, re-run the sparse-checkout cell above.

## 2. Build train/val datasets

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=123,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=123,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)

class_names = train_ds.class_names  # authoritative order used for the labels array
print(class_names)

# No .cache(): caching ~18k decoded 224x224 images in RAM (~11GB) is what was
# blowing out Colab's free-tier system memory and crashing the kernel mid-epoch
# ("could not allocate pinned host of size" -> kernel restart). Disk read +
# decode is cheap next to the GPU compute anyway.
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## 3. Build the model (MobileNetV2 transfer learning)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
# Scales pixels to [-1, 1] - App.js's classifyImage does the same thing
# manually (pixel/127.5 - 1) so the two stay in sync.
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Train (frozen base, fast)

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=8)

## 5. Optional: fine-tune the top of MobileNetV2

Usually improves accuracy a few points. Skip this cell if step 4's validation
accuracy is already good enough.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=5)

## 6. Evaluate

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f"Validation accuracy: {acc:.3f}")

## 7. Convert to TF.js and download

In [ ]:
!pip install -q tensorflowjs

# tensorflowjs's converter still references np.object/np.bool/etc, which
# newer numpy removed (they were deprecated aliases for the builtins).
# Shim them back rather than downgrading numpy, since TensorFlow itself
# depends on the newer numpy already loaded in this session.
import numpy as np
for _name, _builtin in [("object", object), ("bool", bool), ("int", int), ("float", float)]:
    if not hasattr(np, _name):
        setattr(np, _name, _builtin)

import tensorflowjs as tfjs

tfjs.converters.save_keras_model(model, "tfjs_model")

import json
metadata = {
    "modelName": "plantdoctor-tomato-10class",
    "labels": class_names,
    "imageSize": IMG_SIZE,
}
with open("tfjs_model/metadata.json", "w") as f:
    json.dump(metadata, f)

print("Files to copy into public/model/:")
!ls tfjs_model

In [ ]:
!zip -r tfjs_model.zip tfjs_model
from google.colab import files
files.download("tfjs_model.zip")

## 8. Install locally

Unzip `tfjs_model.zip`, then replace everything in `public/model/` in the
PlantDoctor repo with the extracted files (`model.json`, one or more
`group1-shard*.bin` weight files, and `metadata.json`). No app code changes
needed - `App.js` reads `metadata.json`'s `labels` array and resolves
weight file paths from whatever `model.json` says.